# 01 DQA-SoftMoX Judger Probe

Build the first automatic module-wise softmix judger for `G_t / A_t / S_t`.

In [1]:

from __future__ import annotations

import csv
import json
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "dynamic_quality_aware_classwise_aggregation").exists():
    REPO_ROOT = Path("/app/Object_Detection")

JUDGER_ROOT = REPO_ROOT / "dynamic_quality_aware_classwise_aggregation" / "moe_dqa_judger"
RUNNER = JUDGER_ROOT / "scripts" / "run_01_judger_probe.py"
WORKSPACE = JUDGER_ROOT / "output" / "01_judger_probe"
LOG_DIR = JUDGER_ROOT / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("RUNNER:", RUNNER, RUNNER.exists())
print("WORKSPACE:", WORKSPACE)


REPO_ROOT: /app/Object_Detection
RUNNER: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/scripts/run_01_judger_probe.py True
WORKSPACE: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/output/01_judger_probe


## Design

In [2]:

design = pd.DataFrame(
    [
        {
            "block": "goal",
            "setting": "learn automatic G/A/S mixing",
            "detail": "Judger predicts body/head/moe weights over previous global, DQA aggregate, and server repair.",
        },
        {
            "block": "inputs",
            "setting": "reuse 01 artifacts",
            "detail": "Warmup and existing round checkpoints are reused so the judger can be developed without re-running full FL.",
        },
        {
            "block": "module split",
            "setting": "body / head / moe",
            "detail": "Body keeps adaptation, head stays calibrated, MoE keeps specialization.",
        },
        {
            "block": "judger v0",
            "setting": "bootstrap ML model",
            "detail": "Historical round features train a tiny random-forest multi-output regressor for softmix weights.",
        },
        {
            "block": "probe policy",
            "setting": "2 rounds first, then up to 5",
            "detail": "Build/evaluate round1-2 first; if promising, run the same notebook cell for round1-5.",
        },
    ]
)
display(design)


,block,setting,detail
0,goal,learn automatic G/A/S mixing,Judger predicts body/head/moe weights over pre...
1,inputs,reuse 01 artifacts,Warmup and existing round checkpoints are reus...
2,module split,body / head / moe,"Body keeps adaptation, head stays calibrated, ..."
3,judger v0,bootstrap ML model,Historical round features train a tiny random-...
4,probe policy,"2 rounds first, then up to 5","Build/evaluate round1-2 first; if promising, r..."


## Round 1-2 Probe

This is the fast sanity pass. It builds `M_t` checkpoints for the first two rounds.

In [3]:

timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
log_path = LOG_DIR / f"01_judger_probe_r2_{timestamp}.log"
cmd = [
    sys.executable,
    str(RUNNER),
    "--workspace-root", str(WORKSPACE),
    "--history-rounds", "21",
    "--rounds", "1,2",
    "--force",
]
print(" ".join(cmd))
with log_path.open("w", encoding="utf-8") as log:
    proc = subprocess.run(cmd, cwd=REPO_ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    log.write(proc.stdout)
print("returncode:", proc.returncode)
print("log:", log_path)
print(proc.stdout[-4000:])
if proc.returncode != 0:
    raise SystemExit(proc.returncode)


/opt/venv/bin/python3 /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/scripts/run_01_judger_probe.py --workspace-root /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/output/01_judger_probe --history-rounds 21 --rounds 1,2 --force


returncode: 0
log: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/logs/01_judger_probe_r2_20260513_094531.log
[
  {
    "label": "judger_softmix_p1_round001",
    "round": 1.0,
    "path": "/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/output/01_judger_probe/checkpoints/judger_softmix_p1_round001.pt",
    "g_map50": 0.51905,
    "a_proxy_map50": 0.5236449999999999,
    "s_map50": 0.52697,
    "pseudo_mean_score": 0.7738232977461447,
    "pseudo_mean_conf": 0.8057746340450772,
    "pseudo_mean_stability": 0.9567185915628708,
    "pseudo_boxes": 13488.0,
    "pseudo_images": 2910.0,
    "class_entropy_norm": 0.43031203530269735,
    "rare_fraction": 0.0013345195729537367,
    "vehicle_fraction": 0.9670818505338078,
    "expert_entropy_norm": 0.7377531204192945,
    "dead_expert_fraction": 0.25,
    "repair_gain_vs_a": 0.0033250000000001334,
    "repair_gain_vs_g": 0.007920000000000038,
    "body_g": 0.14025632422014267,

In [4]:

weights_csv = WORKSPACE / "stats" / "01_judger_softmix_rounds.csv"
weights = pd.read_csv(weights_csv)
display(weights[[
    "round",
    "g_map50",
    "a_proxy_map50",
    "s_map50",
    "repair_gain_vs_a",
    "body_g", "body_a", "body_s",
    "head_g", "head_a", "head_s",
    "moe_g", "moe_a", "moe_s",
]])
print("report:", WORKSPACE / "01_judger_probe_report.md")


,round,g_map50,a_proxy_map50,s_map50,repair_gain_vs_a,body_g,body_a,body_s,head_g,head_a,head_s,moe_g,moe_a,moe_s
0,1.0,0.51905,0.523645,0.52697,0.003325,0.140256,0.757222,0.102522,0.278822,0.192842,0.528336,0.141096,0.724968,0.133936
1,2.0,0.52697,0.522605,0.52276,0.000155,0.137479,0.762308,0.100213,0.274576,0.209344,0.516080,0.135098,0.732766,0.132136


report: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/output/01_judger_probe/01_judger_probe_report.md


## Extend To Five Rounds

Run this when round 1-2 looks sane. This still reuses existing artifacts; it does not start a new FL training loop.

In [5]:

timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
log_path = LOG_DIR / f"01_judger_probe_r5_{timestamp}.log"
cmd = [
    sys.executable,
    str(RUNNER),
    "--workspace-root", str(WORKSPACE),
    "--history-rounds", "21",
    "--rounds", "1,2,3,4,5",
    "--force",
]
print(" ".join(cmd))
with log_path.open("w", encoding="utf-8") as log:
    proc = subprocess.run(cmd, cwd=REPO_ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    log.write(proc.stdout)
print("returncode:", proc.returncode)
print("log:", log_path)
print(proc.stdout[-4000:])
if proc.returncode != 0:
    raise SystemExit(proc.returncode)

weights = pd.read_csv(WORKSPACE / "stats" / "01_judger_softmix_rounds.csv")
display(weights[[
    "round",
    "g_map50",
    "a_proxy_map50",
    "s_map50",
    "repair_gain_vs_a",
    "body_g", "body_a", "body_s",
    "head_g", "head_a", "head_s",
    "moe_g", "moe_a", "moe_s",
]])


/opt/venv/bin/python3 /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/scripts/run_01_judger_probe.py --workspace-root /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/output/01_judger_probe --history-rounds 21 --rounds 1,2,3,4,5 --force


returncode: 0
log: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/logs/01_judger_probe_r5_20260513_094539.log
pseudo_images": 2528.0,
    "class_entropy_norm": 0.4671684933410906,
    "rare_fraction": 0.0066378356632807,
    "vehicle_fraction": 0.9182339334204969,
    "expert_entropy_norm": 0.7764041285745504,
    "dead_expert_fraction": 0.25,
    "repair_gain_vs_a": 0.00015500000000001624,
    "repair_gain_vs_g": -0.004210000000000047,
    "body_g": 0.13747892313070678,
    "body_a": 0.7623082434947651,
    "body_s": 0.10021283337452795,
    "head_g": 0.274576221282102,
    "head_a": 0.20934382405945923,
    "head_s": 0.5160799546584388,
    "moe_g": 0.13509785092694707,
    "moe_a": 0.732765800406875,
    "moe_s": 0.13213634866617796
  },
  {
    "label": "judger_softmix_p1_round003",
    "round": 3.0,
    "path": "/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/output/01_judger_probe/checkpoints/judger_softmix_p1_rou

,round,g_map50,a_proxy_map50,s_map50,repair_gain_vs_a,body_g,body_a,body_s,head_g,head_a,head_s,moe_g,moe_a,moe_s
0,1.0,0.51905,0.523645,0.52697,0.003325,0.140256,0.757222,0.102522,0.278822,0.192842,0.528336,0.141096,0.724968,0.133936
1,2.0,0.52697,0.522605,0.52276,0.000155,0.137479,0.762308,0.100213,0.274576,0.209344,0.516080,0.135098,0.732766,0.132136
2,3.0,0.52276,0.516175,0.51690,0.000725,0.130612,0.769532,0.099856,0.266937,0.220017,0.513046,0.100511,0.768528,0.130961
3,4.0,0.51690,0.517805,0.50918,-0.008625,0.142785,0.767290,0.089925,0.289897,0.201910,0.508194,0.134083,0.737219,0.128699
4,5.0,0.50918,0.512050,0.50539,-0.006660,0.132600,0.778281,0.089120,0.267890,0.233096,0.499013,0.091688,0.781046,0.127266


## Optional Total Evaluation


Optional total-split evaluation is separated because each checkpoint evaluation is much slower than checkpoint mixing.
Run this cell when the two/five-round weight table looks sane.


In [6]:

timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
log_path = LOG_DIR / f"01_judger_probe_eval_total_{timestamp}.log"
cmd = [
    sys.executable,
    str(RUNNER),
    "--workspace-root", str(WORKSPACE),
    "--history-rounds", "21",
    "--rounds", "1,2",
    "--evaluate",
    "--eval-splits", "total",
    "--val-batch-size", "32",
]
print(" ".join(cmd))
with log_path.open("w", encoding="utf-8") as log:
    proc = subprocess.run(cmd, cwd=REPO_ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    log.write(proc.stdout)
print("returncode:", proc.returncode)
print("log:", log_path)
print(proc.stdout[-4000:])
if proc.returncode != 0:
    raise SystemExit(proc.returncode)

eval_csv = WORKSPACE / "stats" / "01_judger_softmix_eval.csv"
if eval_csv.exists():
    display(pd.read_csv(eval_csv))


/opt/venv/bin/python3 /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/scripts/run_01_judger_probe.py --workspace-root /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/output/01_judger_probe --history-rounds 21 --rounds 1,2 --evaluate --eval-splits total --val-batch-size 32


returncode: 0
log: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/logs/01_judger_probe_eval_total_20260513_094550.log
Saved: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa/output/01_dqa_fedmox_yolo_full/validation_reports/paper_protocol_eval_manifest.json
Saved: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa/output/01_dqa_fedmox_yolo_full/validation_reports/paper_protocol_eval_summary.csv
Saved: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa/output/01_dqa_fedmox_yolo_full/validation_reports/paper_protocol_eval_summary.md
Running eval: /opt/venv/bin/python3 /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/scripts/evaluate_scene_daynight_protocol.py --workspace /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa/output/01_dqa_fedmox_yolo_full --splits total --batch-size 32 --no-plots --checkpoint judger_softmix_p1_rou

,eval_checkpoint_label,eval_checkpoint_path,eval_command,eval_error,eval_images,eval_labels,eval_log_file,eval_map50,eval_map50_95,eval_precision,eval_recall,eval_returncode,eval_split,eval_split_list,eval_status,label,path
0,judger_softmix_p1_round001,/app/Object_Detection/dynamic_quality_aware_cl...,/root/micromamba/envs/al_yolov8/bin/python val...,NaN,9087.0,169419.0,/app/Object_Detection/dynamic_quality_aware_cl...,0.459,0.257,0.712,0.419,0,scene_daynight_total,/app/Object_Detection/dynamic_quality_aware_cl...,ok,judger_softmix_p1_round001,/app/Object_Detection/dynamic_quality_aware_cl...
1,judger_softmix_p1_round002,/app/Object_Detection/dynamic_quality_aware_cl...,/root/micromamba/envs/al_yolov8/bin/python val...,NaN,9087.0,169419.0,/app/Object_Detection/dynamic_quality_aware_cl...,0.461,0.260,0.695,0.430,0,scene_daynight_total,/app/Object_Detection/dynamic_quality_aware_cl...,ok,judger_softmix_p1_round002,/app/Object_Detection/dynamic_quality_aware_cl...
